# Finish the point cloud denoiser - Colab only, no Kaggle

Does everything outstanding in one session:

1. resume training from epoch 46 to 60 (~2 h)
2. benchmark PU-Net **and** PC-Net (~1.5 h)
3. write the final comparison table

**No Kaggle quota is used.** The benchmark data is fetched straight from the
original ScoreDenoise release on Google Drive, and the code comes from your
own Drive or a direct upload.

**Set the runtime first:** Runtime -> Change runtime type -> **T4 GPU**.

Everything is written to `MyDrive/pointdenoise/`, so a disconnect costs at most
one epoch - Colab wipes `/content` when the session ends.


In [ ]:
import os, subprocess, sys

from google.colab import drive
drive.mount("/content/drive")

OUT = "/content/drive/MyDrive/pointdenoise"
os.makedirs(OUT + "/runs", exist_ok=True)
print("outputs ->", OUT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "gdown", "trimesh", "rtree"], check=False)
print("deps installed")


## 1. Code

Put `pointdenoise-code.zip` in `MyDrive/pointdenoise/` before starting, or let
the cell prompt you to upload it. It is 22 KB.


In [ ]:
import zipfile, glob

CODE = None

def looks_like_package(root):
    for base, dirs, _ in os.walk(root):
        if "pointdenoise" in dirs:
            return base
    return None

# already extracted in Drive?
CODE = looks_like_package(OUT)

# a zip sitting in Drive?
if CODE is None:
    zips = glob.glob(OUT + "/*code*.zip") + glob.glob(OUT + "/*.zip")
    for z in zips:
        with zipfile.ZipFile(z) as f:
            if any(n.startswith("pointdenoise/") for n in f.namelist()):
                f.extractall("/content/code")
                CODE = looks_like_package("/content/code")
                print("extracted", z)
                break

# otherwise ask for it
if CODE is None:
    from google.colab import files
    print("Upload pointdenoise-code.zip")
    up = files.upload()
    name = next(iter(up))
    with open("/content/code.zip", "wb") as f:
        f.write(up[name])
    with zipfile.ZipFile("/content/code.zip") as f:
        f.extractall("/content/code")
    CODE = looks_like_package("/content/code")

assert CODE, "could not find the pointdenoise package"
sys.path.insert(0, CODE)
print("package:", CODE)


## 2. Benchmark data

Downloaded from the original ScoreDenoise release, the same files every method
in the comparison table was evaluated on. 224 MB, cached in Drive so it only
happens once.


In [ ]:
import gdown

DATA = OUT + "/data"
os.makedirs(DATA, exist_ok=True)

FILES = {
    "data.zip": "1LdrQhcqDJa-A5Ngiywt0C_KFD60vRqfA",          # PUNet train + test
    "PCNet-Testset.zip": "1RCmwC401IZWgXsUE_DiMG7_HjaI-mGHQ",  # PCNet test
}

for name, fid in FILES.items():
    dest = os.path.join(DATA, name)
    if os.path.exists(dest):
        print(f"{name} already in Drive ({os.path.getsize(dest)/1e6:.0f} MB)")
        continue
    gdown.download(id=fid, output=dest, quiet=False)

# extract once
if not os.path.isdir(os.path.join(DATA, "examples")):
    for name in FILES:
        with zipfile.ZipFile(os.path.join(DATA, name)) as z:
            z.extractall(DATA)
    print("extracted")

print()
for sub in ("examples", "PUNet", "PCNet"):
    print(f"  {sub}: {'ok' if os.path.isdir(os.path.join(DATA, sub)) else 'MISSING'}")


In [ ]:
import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NO GPU - Runtime > Change runtime type > T4")
assert torch.cuda.is_available(), "turn the GPU on before continuing"


## 3. Checkpoint

Resuming from epoch 46 costs ~2 h; starting over costs ~8 h. Put `best.pt` in
`MyDrive/pointdenoise/` beforehand, or let the cell prompt you.


In [ ]:
CKPT = None
for base, _, files_ in os.walk(OUT):
    if "data" in base:
        continue
    for f in ("best.pt", "last.pt"):
        if f in files_:
            CKPT = os.path.join(base, f)
            break
    if CKPT:
        break

if CKPT is None:
    from google.colab import files
    print("No checkpoint in Drive. Upload best.pt, or interrupt to train from scratch.")
    up = files.upload()
    name = next(iter(up))
    CKPT = OUT + "/best.pt"
    with open(CKPT, "wb") as f:
        f.write(up[name])

ck = torch.load(CKPT, map_location="cpu", weights_only=False)
print(f"checkpoint: {CKPT}")
print(f"  epoch {ck['epoch']}, best loss {ck.get('best'):.6f}, kwargs {ck.get('model_kwargs')}")


## 4. Calibrate before spending hours

Confirms the harness reproduces a published number. A run whose harness does
not is comparable to nothing, so this asserts rather than warns.

P2M is expected to fail at 0.17x - our implementation measures something other
than what these papers call P2M, so it is never quoted.


In [ ]:
from pointdenoise.benchmark import calibrate, load_released_set

case = load_released_set(DATA, "PUNet", "sparse", 0.01)
print(f"{case.label}: {len(case.shapes)} shapes")
r = calibrate(case)
for m in ("cd", "p2m"):
    print(f"  {m.upper():<4} ours {r['measured_' + m]:7.3f}  published {r['expected_' + m]:6.2f}"
          f"  ratio {r[m + '_ratio']:.2f}x  {'PASS' if r[m + '_ok'] else 'FAIL'}")
print()
print("quotable:", [m.upper() for m in r["comparable_metrics"]])
assert r["cd_ok"], "CD calibration failed - fix before trusting any number"


## 5. Train

Saves every epoch to Drive. If Colab drops, just re-run this cell - it resumes
from whatever epoch it reached.


In [ ]:
import numpy as np
from pointdenoise.benchmark import load_training_clouds
from pointdenoise.data import Shape
from pointdenoise.engine import train

train_clouds = load_training_clouds(DATA, "PUNet", "sparse")
shapes = [Shape(pts, noise_level=0.02, rng=np.random.default_rng(i))
          for i, (_, pts) in enumerate(train_clouds)]
print(f"{len(shapes)} training shapes, {shapes[0].clean.shape[0]} points each")

EPOCHS = 60
done = ck["epoch"]
print(f"resuming at epoch {done}, {EPOCHS - done} to go (~{(EPOCHS - done) * 8 / 60:.1f} h)")

model, history = train(
    shapes,
    out_dir=OUT + "/runs",
    epochs=EPOCHS,
    batch_size=32,
    points_per_patch=256,
    patches_per_shape=1000,
    lr=1e-3,
    repulsion_weight=0.05,
    noise_range=(0.005, 0.03),   # sampled per patch, not fixed
    model_kwargs={"d_model": 256, "num_heads": 8, "num_layers": 6},
    num_workers=2,
    seed=0,
    resume=CKPT,
)


In [ ]:
import matplotlib.pyplot as plt

fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4))
ep = [h["epoch"] for h in history]
a.plot(ep, [h["total"] for h in history], label="total")
a.plot(ep, [h["chamfer"] for h in history], label="chamfer")
a.set_xlabel("epoch"); a.set_ylabel("loss"); a.legend(); a.grid(alpha=.3)
a.set_title("Training loss")
b.plot(ep, [h["lr"] for h in history], color="tab:orange")
b.set_yscale("log"); b.set_xlabel("epoch"); b.set_ylabel("lr"); b.grid(alpha=.3)
b.set_title("Learning rate")
plt.tight_layout(); plt.savefig(OUT + "/loss.png", dpi=120); plt.show()

print(f"epoch {history[0]['epoch']}: {history[0]['total']:.6f}")
print(f"epoch {history[-1]['epoch']}: {history[-1]['total']:.6f}")


## 6. Benchmark both datasets

PU-Net (20 shapes) and PC-Net (10 shapes), each at 10K/50K points and 1/2/3%
noise. The noisy input is scored alongside every cell, so it stays obvious
whether the model helped rather than only where it ranks.


In [ ]:
import json

import numpy as np
from pointdenoise.benchmark import NOISE_LEVELS, load_released_set, run_case
from pointdenoise.data import Shape
from pointdenoise.engine import denoise_cloud

def denoiser(points):
    shape = Shape(np.asarray(points), noisy=np.asarray(points))
    return denoise_cloud(model, shape, points_per_patch=256, batch_size=128, iters=1)

# Resumable: each (dataset, resolution, noise) cell is written to disk the
# moment it finishes, not batched up for one write at the end. A disconnect
# then costs at most the cell in progress - not every cell computed before it,
# which is what happened last time: 4 cells finished and printed, but nothing
# was on disk to resume from, so a fresh session would have recomputed them.
PARTIAL_PATH = OUT + "/bench_partial.json"

def load_partial():
    if os.path.exists(PARTIAL_PATH):
        with open(PARTIAL_PATH) as f:
            return json.load(f)
    return {}

def save_partial(partial):
    with open(PARTIAL_PATH, "w") as f:
        json.dump(partial, f, indent=2)

partial = load_partial()
results = {}

for dataset in ("PUNet", "PCNet"):
    ds_partial = partial.setdefault(dataset, {})
    scores, baseline = {}, {}

    for resolution in ("sparse", "dense"):
        for noise in NOISE_LEVELS:
            key = f"{resolution}|{noise}"

            if key in ds_partial:
                ours = ds_partial[key]["ours"]
                none = ds_partial[key]["noisy"]
                print(f"{dataset}/{resolution}/{noise:.0%}  (cached) ours CD {ours['cd']:7.4f}  "
                      f"noisy {none['cd']:7.4f}")
            else:
                try:
                    case = load_released_set(DATA, dataset, resolution, noise)
                except (FileNotFoundError, RuntimeError) as e:
                    print(f"skip {dataset}/{resolution}/{noise:.0%}: {e}")
                    continue
                _, ours = run_case(case, denoiser, with_p2m=True)
                _, none = run_case(case, lambda p: p, with_p2m=True)
                gain = (none["cd"] - ours["cd"]) / none["cd"] * 100
                print(f"{dataset}/{resolution}/{noise:.0%}  ours CD {ours['cd']:7.4f}  "
                      f"noisy {none['cd']:7.4f}  {gain:+5.1f}%", flush=True)

                ds_partial[key] = {"ours": ours, "noisy": none}
                save_partial(partial)  # write now - a disconnect costs only this cell

            scores[(resolution, noise)] = ours
            baseline[(resolution, noise)] = none

    if scores:
        results[dataset] = (scores, baseline)

print(f"\ncached at {PARTIAL_PATH} - re-running this cell after a disconnect skips finished cells")


In [ ]:
from pointdenoise.metrics import paper_table

CAVEAT = (
    "CD is calibrated: this harness reproduces the published Bilateral CD to 0.84x\n"
    "on the same shapes, so the CD columns are comparable.\n\n"
    "P2M is NOT calibrated - 0.17x the published value for the same algorithm - so\n"
    "the P2M columns appear because the layout calls for them, not as a claim.\n"
)

out = []
for dataset, (scores, baseline) in results.items():
    published = None if dataset == "PUNet" else {}
    table = paper_table(scores, our_name="Ours", dataset=dataset, published=published)
    print(table); print()
    out += [table, "", f"{dataset} noisy-input baseline (CD x1e-4)"]
    for k, v in baseline.items():
        o = scores[k]["cd"]
        out.append(f"  {k[0]}/{k[1]:.0%}  ours {o:7.4f}  noisy {v['cd']:7.4f}  "
                   f"{(v['cd'] - o) / v['cd'] * 100:+5.1f}%")
    out.append("")

with open(OUT + "/benchmark.txt", "w") as f:
    f.write("\n".join(out) + "\n\n" + CAVEAT)
print("saved", OUT + "/benchmark.txt")


## Done

In `MyDrive/pointdenoise/`:

- `benchmark.txt` - both comparison tables
- `runs/best.pt` - the finished checkpoint
- `runs/history.json` - full loss history
- `loss.png` - training curves

Read the **CD** columns. P2M is not calibrated against the published
definition and is not a claim.
